# `run_folder_pipeline` 一键流程演示

把一个文件夹里的 PDF 依次跑完 `ingest -> requalify -> qualify -> index -> publish -> tree`，可选再用一份题集在进程内问一遍（详见 `docs/enterprise-pdf-rag/testing-and-ingestion.md` 的 `run-folder` 一节）。

**走产品真实路径**：会调用真实的 LLM 与 embedding 后端，产生真实的模型调用（受下方预算限制）。notebook 里不写任何密钥或主机地址，配置全部来自仓库根的 `.env` 或进程环境变量（参见 `.env.example`）。

## 前置条件

- `APP_LLM_API_KEY`、`APP_LLM_BASE_URL`、`APP_LLM_MODEL`：云端 OpenAI 兼容 LLM（ingest / tree / 回答都要用）。
- `APP_EMBEDDING_BASE_URL`、`APP_EMBEDDING_MODEL`、`APP_EMBEDDING_API_KEY`：本地 embedding 服务；pipeline 开跑前会探测一次 `embed_query("preflight")`。
- embedding 走 SSH 隧道时，**先**手动启动：`.venv/bin/python scripts/enterprise_pdf_rag/local_model_tunnel.py start`（pipeline 不会自己起隧道）。
- 题集里若有 `rerank: true` 的用例，还需要 `APP_RERANK_*`。
- 默认输入目录 `data/fixtures/pdf` 是合成 fixture，被 `.gitignore` 的 `data/*` 忽略，不随 git 走；若目标机器上没有，先在仓库根运行 `.venv/bin/python scripts/make_fixtures_pdf.py` 生成（需要 dev 依赖 reportlab）。
- 缺少任何前置时，函数会在花预算之前抛 `PreflightError` 并指出缺哪些变量。

In [ ]:
import os
from pathlib import Path

# ragspine 的 ROOT_DIR 在 import 时从 cwd 向上找 .project-root，
# 所以必须先显式定位仓库根并切过去，再 import 项目代码，不依赖 kernel 启动时的 cwd。
_start = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (_start, *_start.parents) if (p / ".project-root").is_file()), None
)
if REPO_ROOT is None:
    raise RuntimeError(f"从 {_start} 向上找不到 .project-root，请在仓库内启动 notebook")
os.chdir(REPO_ROOT)
print("REPO_ROOT =", REPO_ROOT)

In [ ]:
# ---- 参数（按需修改）----
PDF_FOLDER = REPO_ROOT / "data" / "fixtures" / "pdf"  # 输入 PDF 文件夹（递归查找 *.pdf）
QUESTIONS: Path | None = None  # 可选题集（.json/.jsonl/.csv/.txt 或 nl-answers-gold-v1 .json）
MAX_LIVE_CALLS_PER_PDF = 20  # 每份 PDF 的 ingest 真实模型调用上限
MAX_LIVE_CALLS_TOTAL: int | None = 60  # 整个运行的总调用上限（ingest + tree + 回答共用）；None 不设
BUILD_TREE = False  # True 会为每份已发布文档再做文档树（额外模型调用，另有 TREE_MAX_LIVE_CALLS）
TREE_MAX_LIVE_CALLS = 20
PAGES = "all"  # 如 "1-3"；只影响下游处理，完整源文件仍会保存
INGESTION_ROOT: Path | None = None  # None 用配置默认值（APP_INGESTION_DIR 或 data/ingestion）
REPORT_DIR: Path | None = None  # 给路径则额外写 report.json / report.md

if not PDF_FOLDER.is_dir():
    raise FileNotFoundError(
        f"输入文件夹不存在：{PDF_FOLDER}。请改 PDF_FOLDER，"
        "或运行 .venv/bin/python scripts/make_fixtures_pdf.py 生成合成 fixture"
    )

In [ ]:
from enterprise_pdf_rag.adapters.folder_pipeline import run_folder_pipeline

result = run_folder_pipeline(
    PDF_FOLDER,
    questions=QUESTIONS,
    ingestion_root=INGESTION_ROOT,
    pages=PAGES,
    max_live_calls_per_pdf=MAX_LIVE_CALLS_PER_PDF,
    max_live_calls_total=MAX_LIVE_CALLS_TOTAL,
    build_tree=BUILD_TREE,
    tree_max_live_calls=TREE_MAX_LIVE_CALLS,
    report_dir=REPORT_DIR,
    progress=lambda event, payload: print(event, payload),
)

In [ ]:
print("ok:", result.ok)
print("folder:", result.folder)
print("ingestion_root:", result.ingestion_root)
print("live_calls:", result.live_calls.model_dump())
print("budget_exhausted:", result.budget_exhausted)
print("report_dir:", result.report_dir)
print()
for doc in result.documents:
    published = doc.publication.published_processing_id if doc.publication else None
    members = doc.publication.member_count if doc.publication else None
    print(
        f"{Path(doc.pdf_path).name}: status={doc.status} failed_stage={doc.failed_stage} "
        f"live_calls={doc.live_calls}/{doc.live_call_budget} elapsed_s={doc.elapsed_s} "
        f"published_processing_id={published} member_count={members} error={doc.error}"
    )

if result.eval is not None:
    print()
    print("eval.format:", result.eval.format)
    print("eval.totals:", result.eval.totals)
    print("eval.metrics:", result.eval.metrics)
    for case in result.eval.cases:
        print(case.case_id, case.verdict, "cited_pages=", case.cited_pages, case.failures)